In [2]:
import os
import pandas as pd
from sklearn.metrics import accuracy_score
from tf_slim.metrics import accuracy
from xgboost import XGBRegressor
import numpy as np
import sklearn

import importlib

import prepare_data
importlib.reload(prepare_data)
from prepare_data import *

import sys
sys.path.append('../result_analysis')
import CARE_scores as care
importlib.reload(care)

<module 'CARE_scores' from 'C:\\Users\\olab0\\OneDrive\\Pulpit\\Pulpit_\\Studies\\Informatyka\\ProjektGrupowy\\repo\\Anomaly-Detection-for-Wind-Turbines\\anomaly_detection\\../result_analysis\\CARE_scores.py'>

In [3]:
dataset_dir_path = '../../../data/Care_To_Compare/Wind Farm A/Wind Farm A/datasets'
event_file_path = '../../../data/Care_To_Compare/Wind Farm A/Wind Farm A/event_info.csv'


In [4]:
data = pd.read_csv(event_file_path, sep = ';')
rows_used = [4, 0, 12, 15]

In [5]:
train, prediction = generate_train_prediction(rows_used, data, lagging=1)
prediction = generate_ground_truth(data, prediction)

In [16]:
prediction[0]

,time_stamp,asset_id,id,train_test,status_type_id,sensor_0_avg,sensor_1_avg,sensor_2_avg,wind_speed_3_avg,wind_speed_4_avg,...,sensor_48_lag_1,sensor_49_lag_1,sensor_50_lag_1,sensor_51_lag_1,sensor_52_avg_lag_1,sensor_52_max_lag_1,sensor_52_min_lag_1,sensor_52_std_lag_1,sensor_53_avg_lag_1,is_anomaly
52063,2023-07-28 13:20:00,11,52063,prediction,4,31.0,253.1,-0.4,5.6,5.4,...,-1068.0,0.0,10357.0,-1508.0,6.7,12.8,0.0,5.2,36.0,True
52064,2023-07-28 13:30:00,11,52064,prediction,4,30.0,287.1,18.4,5.7,5.6,...,-36200.0,0.0,41983.0,-36200.0,11.6,12.9,10.9,0.5,36.0,True
52065,2023-07-28 13:40:00,11,52065,prediction,4,30.0,301.6,33.3,6.3,6.2,...,-17412.0,0.0,45924.0,-17412.0,11.7,13.9,10.9,0.7,35.0,True
52066,2023-07-28 13:50:00,11,52066,prediction,4,30.0,259.0,-9.3,6.3,6.1,...,-18263.0,0.0,65606.0,-18263.0,12.4,14.2,11.0,0.9,35.0,True
52067,2023-07-28 14:00:00,11,52067,prediction,4,30.0,295.8,19.8,5.7,5.6,...,-15416.0,0.0,60958.0,-15416.0,12.3,14.0,11.2,0.6,35.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54353,2023-08-13 12:40:00,11,54353,prediction,3,27.0,94.7,-9.5,16.2,14.8,...,-36056.0,0.0,333259.0,-36056.0,14.9,15.7,14.3,0.2,30.0,False
54354,2023-08-13 12:50:00,11,54354,prediction,3,27.0,99.8,-10.7,15.7,14.2,...,-36016.0,0.0,333802.0,-36016.0,14.9,15.5,14.4,0.2,30.0,False
54355,2023-08-13 13:00:00,11,54355,prediction,3,28.0,98.2,-6.0,14.6,13.5,...,-35933.0,0.0,333280.0,-35933.0,14.9,15.6,14.3,0.2,31.0,False
54356,2023-08-13 13:10:00,11,54356,prediction,3,28.0,105.0,-6.2,15.2,14.1,...,-36010.0,0.0,332927.0,-36010.0,14.9,15.7,14.4,0.2,31.0,False


In [21]:
#another approach - train on one, test on one
errors = {}
errors_scaled = {}

errors_pred = {}
errors_scaled_pred = {}
for key in rows_used[:]:
    print("KEY: ", key)
    train_set = train[key]
    prediction_set = prediction[key]
    
    X_train = train_set
    X_pred = prediction_set.drop(['is_anomaly'], axis=1) #prediction data
    y_train = [False for i in range(X_train.shape[0])]

    X_train = X_train.drop(['asset_id_lag_1','id_lag_1', 'train_test_lag_1', 'status_type_id_lag_1', 'time_stamp_lag_1', 'asset_id','id', 'train_test', 'status_type_id', 'time_stamp'], axis=1)
    X_pred = X_pred.drop(['asset_id_lag_1','id_lag_1', 'train_test_lag_1', 'status_type_id_lag_1', 'time_stamp_lag_1', 'asset_id','id', 'train_test', 'status_type_id', 'time_stamp'], axis=1)
    errors_ = pd.DataFrame(index = train_set.index)
    errors_pred_ = pd.DataFrame(index = prediction_set.index)
    
    for col in X_train.columns[:]:
        print("COL: ", col)
        X_train_ = X_train.dropna(subset=[col]) 
        
        X_train_ = X_train_.drop(columns=[col])
        y_train_ = X_train[col]
        
        X_pred_ = X_pred.dropna(subset=[col])   #'prediction' data
        X_pred_ = X_pred_.drop(columns=[col])
        y_pred_ = X_pred[col]
        
        
        model = XGBRegressor(n_estimators=100, verbosity=0)
        model.fit(X_train_, y_train_)
        
        y_pred = model.predict(X_train_)  #predicting for 'train' data
        y_pred_pred = model.predict(X_pred_) #predicting for 'prediction' data
        
        errors_[col] = np.abs(y_train_ - y_pred)
        errors_pred_[col] = np.abs(y_pred_ - y_pred_pred)
    errors_.to_csv('errors'+str(key)+'.csv')
    errors[key] = errors_
    errors_scaled[key] = (errors[key] - errors[key].mean()) / errors[key].std()
    
    errors_pred_.to_csv('errors_pred'+str(key)+'.csv')
    errors_pred[key] = errors_pred_
    errors_scaled_pred[key] = (errors_pred[key] - errors_pred[key].mean()) / errors_pred[key].std()
    
    
    

KEY:  4
COL:  sensor_0_avg
COL:  sensor_1_avg
COL:  sensor_2_avg
COL:  wind_speed_3_avg
COL:  wind_speed_4_avg
COL:  wind_speed_3_max
COL:  wind_speed_3_min
COL:  wind_speed_3_std
COL:  sensor_5_avg
COL:  sensor_5_max
COL:  sensor_5_min
COL:  sensor_5_std
COL:  sensor_6_avg
COL:  sensor_7_avg
COL:  sensor_8_avg
COL:  sensor_9_avg
COL:  sensor_10_avg
COL:  sensor_11_avg
COL:  sensor_12_avg
COL:  sensor_13_avg
COL:  sensor_14_avg
COL:  sensor_15_avg
COL:  sensor_16_avg
COL:  sensor_17_avg
COL:  sensor_18_avg
COL:  sensor_18_max
COL:  sensor_18_min
COL:  sensor_18_std
COL:  sensor_19_avg
COL:  sensor_20_avg
COL:  sensor_21_avg
COL:  sensor_22_avg
COL:  sensor_23_avg
COL:  sensor_24_avg
COL:  sensor_25_avg
COL:  sensor_26_avg
COL:  reactive_power_27_avg
COL:  reactive_power_27_max
COL:  reactive_power_27_min
COL:  reactive_power_27_std
COL:  reactive_power_28_avg
COL:  reactive_power_28_max
COL:  reactive_power_28_min
COL:  reactive_power_28_std
COL:  power_29_avg
COL:  power_29_max
COL:  

C:\Users\olab0\AppData\Local\Temp\ipykernel_14716\478215523.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  errors_[col] = np.abs(y_train_ - y_pred)
C:\Users\olab0\AppData\Local\Temp\ipykernel_14716\478215523.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  errors_pred_[col] = np.abs(y_pred_ - y_pred_pred)


COL:  sensor_14_avg_lag_1


C:\Users\olab0\AppData\Local\Temp\ipykernel_14716\478215523.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  errors_[col] = np.abs(y_train_ - y_pred)
C:\Users\olab0\AppData\Local\Temp\ipykernel_14716\478215523.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  errors_pred_[col] = np.abs(y_pred_ - y_pred_pred)


COL:  sensor_15_avg_lag_1


...

C:\Users\olab0\AppData\Local\Temp\ipykernel_14716\478215523.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  errors_[col] = np.abs(y_train_ - y_pred)
C:\Users\olab0\AppData\Local\Temp\ipykernel_14716\478215523.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  errors_pred_[col] = np.abs(y_pred_ - y_pred_pred)


COL:  sensor_52_std_lag_1


C:\Users\olab0\AppData\Local\Temp\ipykernel_14716\478215523.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  errors_[col] = np.abs(y_train_ - y_pred)
C:\Users\olab0\AppData\Local\Temp\ipykernel_14716\478215523.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  errors_pred_[col] = np.abs(y_pred_ - y_pred_pred)


COL:  sensor_53_avg_lag_1


C:\Users\olab0\AppData\Local\Temp\ipykernel_14716\478215523.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  errors_[col] = np.abs(y_train_ - y_pred)
C:\Users\olab0\AppData\Local\Temp\ipykernel_14716\478215523.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  errors_pred_[col] = np.abs(y_pred_ - y_pred_pred)


In [24]:
for key in rows_used[:]:
   errors_scaled[key] = errors_scaled[key].dropna(axis=1)
   errors_scaled_pred[key] = errors_scaled_pred[key].dropna(axis=1)


In [25]:
errors_scaled[0]

,sensor_0_avg,sensor_1_avg,sensor_2_avg,wind_speed_3_avg,wind_speed_4_avg,wind_speed_3_max,wind_speed_3_min,wind_speed_3_std,sensor_5_avg,sensor_5_max,...,sensor_45_lag_1,sensor_47_lag_1,sensor_48_lag_1,sensor_50_lag_1,sensor_51_lag_1,sensor_52_avg_lag_1,sensor_52_max_lag_1,sensor_52_min_lag_1,sensor_52_std_lag_1,sensor_53_avg_lag_1
1,0.049034,-0.446918,6.930276,-0.234347,-0.004806,-0.043736,0.318057,-1.073876,0.784221,-0.146551,...,-0.766942,-0.207146,-0.729866,-0.783941,0.595659,-0.378864,-0.158682,-0.539803,-0.042642,-0.094193
2,0.180691,-0.187599,-0.607509,0.787718,0.663991,-0.391939,0.673441,0.391377,0.220325,-0.468059,...,-0.730053,3.119966,-0.709153,-0.766799,0.171631,-0.503294,-0.737339,-0.644989,0.057121,0.439705
3,1.243783,0.036984,2.475197,-0.637232,0.465824,-0.415202,-0.237800,0.143313,2.053190,0.739288,...,-0.754070,-0.305542,-0.103638,-0.630425,-0.032341,-0.397296,-0.226201,-0.422933,-0.221123,-0.262527
4,-0.010649,-0.459735,-0.463771,0.299798,-0.557879,-0.496508,-0.533584,1.124148,-0.539446,-0.103879,...,0.116299,-0.399187,1.248631,-0.058753,0.265876,0.855182,-0.363536,-0.485349,-0.207782,1.459414
5,-0.343007,-0.347015,-0.524843,0.864709,-0.780798,-0.596766,1.684097,1.774336,-0.060196,0.431901,...,-0.571644,-0.420595,0.286451,-0.141319,0.377564,-0.373482,-0.540102,-0.150099,2.578377,-0.866125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52058,0.003084,1.148671,2.267829,-0.745270,-0.881859,-0.585531,-0.562501,-0.292029,-0.506540,-0.674995,...,-0.818626,2.389177,-0.727109,-0.804742,-0.595195,0.188256,-0.403131,-0.658854,0.727452,1.209056
52059,0.215054,0.819906,0.129418,-0.784145,-0.856830,-0.763844,-0.673030,-0.683168,-0.506540,-0.717975,...,-0.798299,1.095978,-0.764230,-0.847374,-0.660879,7.893720,2.765545,-0.599670,3.227655,0.775125
52060,0.761608,0.907198,0.805432,-0.794015,-0.849549,-0.697235,-0.702993,-0.587563,-0.500681,-0.724527,...,-0.823123,-0.387809,-0.764230,-0.816488,-0.179767,2.722716,1.260927,-0.605143,4.547884,1.354566
52061,-0.125817,1.412090,1.378727,-0.851753,-0.856895,-0.485928,-0.661061,-0.348173,-0.506540,-0.473701,...,-0.810172,0.797224,-0.759854,-0.789883,0.205941,1.139150,0.867752,-0.617952,0.924494,1.392277


In [15]:
errors_scaled[0].mean(axis=1)

1        0.107669
2        0.068365
3        0.070134
4        0.071415
5        0.112277
           ...   
52058   -0.012180
52059    0.018488
52060    0.002887
52061   -0.113025
52062   -0.039405
Length: 52062, dtype: float64

In [37]:
for key in rows_used:
    anomaly_scores_train = errors_scaled[key].mean(axis=1)
    anomaly_scores_pred = errors_scaled_pred[key].mean(axis=1)
    
    threshold = np.percentile(anomaly_scores_train, 90)
    prediction[key]['anomaly_score'] = anomaly_scores_pred
    prediction[key]['is_anomaly_predicted'] = prediction[key]['anomaly_score'] > threshold
    

In [38]:
prediction[0]

,time_stamp,asset_id,id,train_test,status_type_id,sensor_0_avg,sensor_1_avg,sensor_2_avg,wind_speed_3_avg,wind_speed_4_avg,...,sensor_50_lag_1,sensor_51_lag_1,sensor_52_avg_lag_1,sensor_52_max_lag_1,sensor_52_min_lag_1,sensor_52_std_lag_1,sensor_53_avg_lag_1,is_anomaly,anomaly_score,is_anomaly_predicted
52063,2023-07-28 13:20:00,11,52063,prediction,4,31.0,253.1,-0.4,5.6,5.4,...,10357.0,-1508.0,6.7,12.8,0.0,5.2,36.0,True,0.919229,True
52064,2023-07-28 13:30:00,11,52064,prediction,4,30.0,287.1,18.4,5.7,5.6,...,41983.0,-36200.0,11.6,12.9,10.9,0.5,36.0,True,0.602717,True
52065,2023-07-28 13:40:00,11,52065,prediction,4,30.0,301.6,33.3,6.3,6.2,...,45924.0,-17412.0,11.7,13.9,10.9,0.7,35.0,True,-0.065030,False
52066,2023-07-28 13:50:00,11,52066,prediction,4,30.0,259.0,-9.3,6.3,6.1,...,65606.0,-18263.0,12.4,14.2,11.0,0.9,35.0,True,-0.009270,False
52067,2023-07-28 14:00:00,11,52067,prediction,4,30.0,295.8,19.8,5.7,5.6,...,60958.0,-15416.0,12.3,14.0,11.2,0.6,35.0,True,-0.035956,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54353,2023-08-13 12:40:00,11,54353,prediction,3,27.0,94.7,-9.5,16.2,14.8,...,333259.0,-36056.0,14.9,15.7,14.3,0.2,30.0,False,0.147589,False
54354,2023-08-13 12:50:00,11,54354,prediction,3,27.0,99.8,-10.7,15.7,14.2,...,333802.0,-36016.0,14.9,15.5,14.4,0.2,30.0,False,0.198286,False
54355,2023-08-13 13:00:00,11,54355,prediction,3,28.0,98.2,-6.0,14.6,13.5,...,333280.0,-35933.0,14.9,15.6,14.3,0.2,31.0,False,0.118529,False
54356,2023-08-13 13:10:00,11,54356,prediction,3,28.0,105.0,-6.2,15.2,14.1,...,332927.0,-36010.0,14.9,15.7,14.4,0.2,31.0,False,0.013291,False


In [39]:
rows_anomalous = [4, 0]
rows_normal = [12, 15]

In [40]:
cov, ear = [], []
for key in rows_anomalous:
    y_true = prediction[key]['is_anomaly']
    y_pred = prediction[key]['is_anomaly_predicted']
    
    cov.append(care.CARE_coverage(y_true, y_pred))
    
    
    ear.append(care.CARE_earliness(y_true, y_pred))

print('Coverage: ' + str(sum(cov) / len(cov)))
print('Earliness: ' + str(sum(ear) / len(ear)))

Coverage: 0.27140550063391433
Earliness: 0.07746695489832067


In [41]:
acc = []
for key in rows_normal:
    y_true = prediction[key]['is_anomaly']
    y_pred = prediction[key]['is_anomaly_predicted']
    
    acc.append(care.CARE_accuracy(y_true, y_pred))

print('Accuracy: ' + str(sum(acc) / len(acc)))

Accuracy: 0.8909673413698485


In [42]:
rel = 0
event_label_true = []
event_label_pred = []
for key in rows_used:
    if key in rows_anomalous:
        event_label_true.append(True)
    else:
        event_label_true.append(False)
    
    status_id = prediction[key]['status_type_id']
    
    y_pred = prediction[key]['is_anomaly_predicted']
    event_label_pred.append(care.compute_event_label(y_pred, status_id, threshold=72))
    
rel = care.CARE_reliability(event_label_true, event_label_pred)
print('Reliability: ', rel)

    

Reliability:  0


C:\Users\olab0\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Old version below

In [ ]:
df_full_x["anomaly_score"] = errors_scaled.mean(axis=1)

threshold = np.percentile(df_full_x.loc[:(df_train_x.shape[0]), "anomaly_score"], 97)
df_full_x["is_anomaly_predicted"] = df_full_x["anomaly_score"] > threshold

In [26]:
train_flat = (pd.concat([train[0], train[4], train[12], train[15]])).drop(['asset_id','id', 'train_test', 'status_type_id' ], axis=1)
train_flat['is_anomaly'] = False;


In [27]:
train_flat = train_flat.drop(['asset_id_lag_1','id_lag_1', 'train_test_lag_1', 'status_type_id_lag_1', 'time_stamp_lag_1'], axis=1)
"""
train_flat = train_flat.drop(['asset_id_lag_2','id_lag_2', 'train_test_lag_2', 'status_type_id_lag_2', 'time_stamp_lag_2'], axis=1)
train_flat = train_flat.drop(['asset_id_lag_3','id_lag_3', 'train_test_lag_3', 'status_type_id_lag_3', 'time_stamp_lag_3'], axis=1)
train_flat = train_flat.drop(['asset_id_lag_4','id_lag_4', 'train_test_lag_4', 'status_type_id_lag_4', 'time_stamp_lag_4'], axis=1)
train_flat = train_flat.drop(['asset_id_lag_5','id_lag_5', 'train_test_lag_5', 'status_type_id_lag_5', 'time_stamp_lag_5'], axis=1)
"""

"\ntrain_flat = train_flat.drop(['asset_id_lag_2','id_lag_2', 'train_test_lag_2', 'status_type_id_lag_2', 'time_stamp_lag_2'], axis=1)\ntrain_flat = train_flat.drop(['asset_id_lag_3','id_lag_3', 'train_test_lag_3', 'status_type_id_lag_3', 'time_stamp_lag_3'], axis=1)\ntrain_flat = train_flat.drop(['asset_id_lag_4','id_lag_4', 'train_test_lag_4', 'status_type_id_lag_4', 'time_stamp_lag_4'], axis=1)\ntrain_flat = train_flat.drop(['asset_id_lag_5','id_lag_5', 'train_test_lag_5', 'status_type_id_lag_5', 'time_stamp_lag_5'], axis=1)\n"

In [28]:
prediction_flat = (pd.concat([prediction[0], prediction[4], prediction[12], prediction[15]])).drop(['asset_id','id', 'train_test', 'status_type_id' ], axis=1)

In [29]:
prediction_flat =prediction_flat.drop(['asset_id_lag_1','id_lag_1', 'train_test_lag_1', 'status_type_id_lag_1', 'time_stamp_lag_1'], axis=1)
"""
prediction_flat = prediction_flat.drop(['asset_id_lag_2','id_lag_2', 'train_test_lag_2', 'status_type_id_lag_2', 'time_stamp_lag_2'], axis=1)
prediction_flat = prediction_flat.drop(['asset_id_lag_3','id_lag_3', 'train_test_lag_3', 'status_type_id_lag_3', 'time_stamp_lag_3'], axis=1)
prediction_flat = prediction_flat.drop(['asset_id_lag_4','id_lag_4', 'train_test_lag_4', 'status_type_id_lag_4', 'time_stamp_lag_4'], axis=1)
prediction_flat = prediction_flat.drop(['asset_id_lag_5','id_lag_5', 'train_test_lag_5', 'status_type_id_lag_5', 'time_stamp_lag_5'], axis=1)
"""

"\nprediction_flat = prediction_flat.drop(['asset_id_lag_2','id_lag_2', 'train_test_lag_2', 'status_type_id_lag_2', 'time_stamp_lag_2'], axis=1)\nprediction_flat = prediction_flat.drop(['asset_id_lag_3','id_lag_3', 'train_test_lag_3', 'status_type_id_lag_3', 'time_stamp_lag_3'], axis=1)\nprediction_flat = prediction_flat.drop(['asset_id_lag_4','id_lag_4', 'train_test_lag_4', 'status_type_id_lag_4', 'time_stamp_lag_4'], axis=1)\nprediction_flat = prediction_flat.drop(['asset_id_lag_5','id_lag_5', 'train_test_lag_5', 'status_type_id_lag_5', 'time_stamp_lag_5'], axis=1)\n"

In [30]:
df_full = pd.concat([train_flat, prediction_flat])
df_full_x, df_full_y = df_full.drop(['is_anomaly', 'time_stamp'], axis = 1), df_full['is_anomaly']
df_train_x, df_train_y = train_flat.drop(['is_anomaly', 'time_stamp'], axis = 1), train_flat['is_anomaly']

In [15]:
df_full_x = df_full_x.reset_index().drop(['index'],axis=1)
df_train_x = df_train_x.reset_index().drop(['index'],axis=1)

In [16]:
df_full = df_full.reset_index().drop(['index'],axis=1)


In [17]:
errors = pd.DataFrame(index=df_full.index)

for col in df_full_x.columns[:]:
    print(col)
    Xy_train = df_train_x.dropna(subset=[col]) 
    X_train = Xy_train.drop(columns=[col])
    y_train = Xy_train[col]

    X_full = df_full_x.drop(columns=[col])
    y_full = df_full_x[col]

    model = XGBRegressor(n_estimators=100, verbosity=0)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_full)
    errors[col] = np.abs(y_full - y_pred)
    
    

sensor_0_avg
sensor_1_avg
sensor_2_avg
wind_speed_3_avg
wind_speed_4_avg
wind_speed_3_max
wind_speed_3_min
wind_speed_3_std
sensor_5_avg
sensor_5_max
sensor_5_min
sensor_5_std
sensor_6_avg
sensor_7_avg
sensor_8_avg
sensor_9_avg
sensor_10_avg
sensor_11_avg
sensor_12_avg
sensor_13_avg
sensor_14_avg
sensor_15_avg
sensor_16_avg
sensor_17_avg
sensor_18_avg
sensor_18_max
sensor_18_min
sensor_18_std
sensor_19_avg
sensor_20_avg
sensor_21_avg
sensor_22_avg
sensor_23_avg
sensor_24_avg
sensor_25_avg
sensor_26_avg
reactive_power_27_avg
reactive_power_27_max
reactive_power_27_min
reactive_power_27_std
reactive_power_28_avg
reactive_power_28_max
reactive_power_28_min
reactive_power_28_std
power_29_avg
power_29_max
power_29_min
power_29_std
power_30_avg
power_30_max
power_30_min
power_30_std
sensor_31_avg
sensor_31_max
sensor_31_min
sensor_31_std
sensor_32_avg
sensor_33_avg
sensor_34_avg
sensor_35_avg
sensor_36_avg
sensor_37_avg
sensor_38_avg
sensor_39_avg
sensor_40_avg
sensor_41_avg
sensor_42_avg
se

C:\Users\olab0\AppData\Local\Temp\ipykernel_16056\653872583.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  errors[col] = np.abs(y_full - y_pred)


sensor_14_avg_lag_1


C:\Users\olab0\AppData\Local\Temp\ipykernel_16056\653872583.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  errors[col] = np.abs(y_full - y_pred)


sensor_15_avg_lag_1


...

C:\Users\olab0\AppData\Local\Temp\ipykernel_16056\653872583.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  errors[col] = np.abs(y_full - y_pred)


sensor_52_std_lag_1


C:\Users\olab0\AppData\Local\Temp\ipykernel_16056\653872583.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  errors[col] = np.abs(y_full - y_pred)


sensor_53_avg_lag_1


C:\Users\olab0\AppData\Local\Temp\ipykernel_16056\653872583.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  errors[col] = np.abs(y_full - y_pred)


In [21]:
errors.to_csv('errors_1_lag_not_scaled.csv')

In [50]:
df_train_x.shape

(208616, 81)

In [51]:
prediction_flat.shape

(11028, 83)

In [52]:
df_full.shape

(219644, 83)

In [62]:
(df_full[(df_train.shape[0]):])['is_anomaly'].value_counts('True')

is_anomaly
False    0.634929
True     0.365071
Name: proportion, dtype: float64

In [93]:
df_full['is_anomaly_predicted'].value_counts('True')

is_anomaly_predicted
False    0.948981
True     0.051019
Name: proportion, dtype: float64

In [94]:
df_full['is_anomaly'].value_counts('True')

is_anomaly
False    0.98167
True     0.01833
Name: proportion, dtype: float64

In [211]:
df_full["anomaly_score"] = errors.mean(axis=1)  # or use max(), median(), etc.

# Set threshold — top 5% highest errors
threshold = np.percentile(df_full.reset_index().loc[:(df_train.shape[0]), "anomaly_score"], 97)
df_full["is_anomaly_predicted"] = df_full["anomaly_score"] > threshold

In [18]:
errors = errors.reset_index()


In [22]:
errors_scaled = (errors - errors.mean()) / errors.std()
#df["anomaly_score"] = errors_scaled.mean(axis=1)

In [189]:
errors_scaled

,index,sensor_1_avg,sensor_2_avg,wind_speed_3_avg,wind_speed_4_avg,wind_speed_3_max,wind_speed_3_min,wind_speed_3_std,sensor_5_avg,sensor_5_max,...,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51,sensor_52_avg,sensor_52_max,sensor_52_min,sensor_52_std,sensor_53_avg
0,-1.731828,0.357526,-0.186611,-0.534987,1.470731,-0.466350,-0.796176,1.250777,18.476193,2.763570,...,2.316651,-0.666907,NaN,-0.706183,-0.430515,0.313007,3.650891,-0.589312,1.463948,1.396881
1,-1.731765,0.615820,0.800519,-0.405667,0.867587,0.159219,0.916786,-0.147075,-0.334105,-0.616121,...,0.068733,-0.574963,NaN,-0.797086,-0.446243,-0.692886,-0.767139,-0.562378,-0.383815,0.970798
2,-1.731702,-0.064900,1.462154,0.999443,0.223429,-0.687093,0.712488,4.058986,-0.192831,-0.614334,...,-0.254813,-0.574963,NaN,-0.751206,-0.014046,-0.530927,-0.732152,-0.562378,-0.210625,0.388250
3,-1.731639,0.066181,0.363690,-0.710587,1.862845,-0.619714,0.013317,1.935551,2.771804,0.929704,...,0.257298,-0.438369,NaN,0.336741,0.715864,1.207570,-0.309589,-0.586264,-0.233493,-0.532633
4,-1.731575,-0.117388,-0.413565,0.063454,0.125917,-0.809980,-0.721216,0.908434,0.178560,-0.478338,...,-0.272776,-0.071446,NaN,0.624097,0.242707,-0.685598,-0.199990,-0.464024,2.964130,-0.832832
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219639,1.767719,-0.352184,-0.345954,-0.405441,1.097836,1.803455,-0.248189,0.994721,0.171255,-0.580078,...,-0.400603,-0.576775,NaN,2.653483,-0.439922,-0.185827,0.280252,0.089330,-0.400265,-1.008515
219640,1.767782,-0.248800,-0.264332,0.247053,-0.317176,-0.821229,-0.665614,0.050466,-0.529660,-0.400042,...,-0.397675,-0.374088,NaN,0.082709,-0.457240,-0.160772,0.583365,1.662561,-0.371823,-1.174342
219641,1.767846,-0.234045,-0.174188,0.241482,0.188502,0.487447,0.972122,1.674285,-0.493020,-0.463882,...,-0.400603,-0.670818,NaN,-0.823255,-0.516416,-0.598099,-0.617178,0.163623,-0.294026,-0.680324
219642,1.767909,-0.174909,-0.418315,0.018723,-0.601856,-0.835849,1.004487,-1.092786,-0.489561,-0.189263,...,-0.397675,-0.090538,NaN,-0.265016,0.309238,0.250633,-0.563820,-0.617681,-0.398934,0.912614


In [174]:
df_full_x.index

RangeIndex(start=0, stop=219644, step=1)

In [176]:
errors_scaled.index

Index([    0,     1,     2,     3,     4,     5,     6,     7,     8,     9,
       ...
       55477, 55478, 55479, 55480, 55481, 55482, 55483, 55484, 55485, 55486],
      dtype='int64', length=219644)

In [182]:
df_full_x.columns

Index(['sensor_0_avg', 'sensor_1_avg', 'sensor_2_avg', 'wind_speed_3_avg',
       'wind_speed_4_avg', 'wind_speed_3_max', 'wind_speed_3_min',
       'wind_speed_3_std', 'sensor_5_avg', 'sensor_5_max', 'sensor_5_min',
       'sensor_5_std', 'sensor_6_avg', 'sensor_7_avg', 'sensor_8_avg',
       'sensor_9_avg', 'sensor_10_avg', 'sensor_11_avg', 'sensor_12_avg',
       'sensor_13_avg', 'sensor_14_avg', 'sensor_15_avg', 'sensor_16_avg',
       'sensor_17_avg', 'sensor_18_avg', 'sensor_18_max', 'sensor_18_min',
       'sensor_18_std', 'sensor_19_avg', 'sensor_20_avg', 'sensor_21_avg',
       'sensor_22_avg', 'sensor_23_avg', 'sensor_24_avg', 'sensor_25_avg',
       'sensor_26_avg', 'reactive_power_27_avg', 'reactive_power_27_max',
       'reactive_power_27_min', 'reactive_power_27_std',
       'reactive_power_28_avg', 'reactive_power_28_max',
       'reactive_power_28_min', 'reactive_power_28_std', 'power_29_avg',
       'power_29_max', 'power_29_min', 'power_29_std', 'power_30_avg',
      

In [23]:
df_full_x["anomaly_score"] = errors_scaled.mean(axis=1)

threshold = np.percentile(df_full_x.loc[:(df_train_x.shape[0]), "anomaly_score"], 97)
df_full_x["is_anomaly_predicted"] = df_full_x["anomaly_score"] > threshold

In [186]:
errors_scaled

,sensor_1_avg,sensor_2_avg,wind_speed_3_avg,wind_speed_4_avg,wind_speed_3_max,wind_speed_3_min,wind_speed_3_std,sensor_5_avg,sensor_5_max,sensor_5_min,...,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51,sensor_52_avg,sensor_52_max,sensor_52_min,sensor_52_std,sensor_53_avg
0,0.357526,-0.186611,-0.534987,1.470731,-0.466350,-0.796176,1.250777,18.476193,2.763570,2.598983,...,2.316651,-0.666907,NaN,-0.706183,-0.430515,0.313007,3.650891,-0.589312,1.463948,1.396881
1,0.615820,0.800519,-0.405667,0.867587,0.159219,0.916786,-0.147075,-0.334105,-0.616121,-0.592235,...,0.068733,-0.574963,NaN,-0.797086,-0.446243,-0.692886,-0.767139,-0.562378,-0.383815,0.970798
2,-0.064900,1.462154,0.999443,0.223429,-0.687093,0.712488,4.058986,-0.192831,-0.614334,-0.389008,...,-0.254813,-0.574963,NaN,-0.751206,-0.014046,-0.530927,-0.732152,-0.562378,-0.210625,0.388250
3,0.066181,0.363690,-0.710587,1.862845,-0.619714,0.013317,1.935551,2.771804,0.929704,-0.500524,...,0.257298,-0.438369,NaN,0.336741,0.715864,1.207570,-0.309589,-0.586264,-0.233493,-0.532633
4,-0.117388,-0.413565,0.063454,0.125917,-0.809980,-0.721216,0.908434,0.178560,-0.478338,-0.193173,...,-0.272776,-0.071446,NaN,0.624097,0.242707,-0.685598,-0.199990,-0.464024,2.964130,-0.832832
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55482,-0.352184,-0.345954,-0.405441,1.097836,1.803455,-0.248189,0.994721,0.171255,-0.580078,-0.662622,...,-0.400603,-0.576775,NaN,2.653483,-0.439922,-0.185827,0.280252,0.089330,-0.400265,-1.008515
55483,-0.248800,-0.264332,0.247053,-0.317176,-0.821229,-0.665614,0.050466,-0.529660,-0.400042,-0.657554,...,-0.397675,-0.374088,NaN,0.082709,-0.457240,-0.160772,0.583365,1.662561,-0.371823,-1.174342
55484,-0.234045,-0.174188,0.241482,0.188502,0.487447,0.972122,1.674285,-0.493020,-0.463882,0.695970,...,-0.400603,-0.670818,NaN,-0.823255,-0.516416,-0.598099,-0.617178,0.163623,-0.294026,-0.680324
55485,-0.174909,-0.418315,0.018723,-0.601856,-0.835849,1.004487,-1.092786,-0.489561,-0.189263,-0.630707,...,-0.397675,-0.090538,NaN,-0.265016,0.309238,0.250633,-0.563820,-0.617681,-0.398934,0.912614


In [274]:
y_pred = df_full_x['is_anomaly_predicted']
y_true = df_full_y

In [25]:
y_pred_test = df_full_x.loc[(df_train_x.shape[0]):,'is_anomaly_predicted']
y_true_test = df_full_y[(df_train_x.shape[0]):]

In [192]:
df_full_x

,sensor_0_avg,sensor_1_avg,sensor_2_avg,wind_speed_3_avg,wind_speed_4_avg,wind_speed_3_max,wind_speed_3_min,wind_speed_3_std,sensor_5_avg,sensor_5_max,...,sensor_49,sensor_50,sensor_51,sensor_52_avg,sensor_52_max,sensor_52_min,sensor_52_std,sensor_53_avg,anomaly_score,is_anomaly_predicted
0,31.0,152.0,48.7,3.9,3.9,8.0,0.6,0.9,70.5,86.2,...,0.0,-1185.0,-2090.0,0.4,2.6,0.0,0.8,34.0,0.757644,True
1,31.0,86.1,150.9,6.0,6.0,9.9,0.6,1.4,86.0,86.0,...,0.0,-1050.0,-1627.0,0.0,0.0,0.0,0.0,34.0,0.348056,False
2,31.0,115.2,69.6,6.3,6.3,10.6,0.8,1.3,86.0,86.1,...,0.0,-1043.0,-1624.0,0.0,0.0,0.0,0.0,34.0,0.101935,False
3,32.0,129.3,-29.1,6.0,5.9,12.4,1.7,1.4,13.6,86.0,...,0.0,40124.0,-9753.0,9.5,14.0,0.0,4.8,34.0,0.758396,True
4,32.0,137.7,26.4,7.1,6.9,13.7,1.7,1.7,-1.9,0.5,...,0.0,99360.0,-25215.0,13.1,14.9,10.8,1.3,35.0,0.356465,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219639,19.0,97.3,0.7,8.4,8.2,16.3,1.7,1.3,-2.2,-1.3,...,0.0,159979.0,-28066.0,14.4,14.8,13.0,0.3,21.0,0.044690,False
219640,19.0,91.5,-5.0,9.3,8.9,16.0,3.7,1.3,-2.0,0.7,...,0.0,204068.0,-28949.0,14.7,14.9,14.1,0.1,21.0,-0.060051,False
219641,19.0,83.3,-13.2,8.3,8.1,13.4,3.7,1.1,-2.2,-1.7,...,0.0,155666.0,-26837.0,14.4,14.8,13.4,0.3,21.0,-0.091754,False
219642,19.0,111.2,14.6,8.3,8.2,14.6,3.6,1.1,-2.2,-1.6,...,0.0,160533.0,-25319.0,14.4,14.9,13.4,0.3,21.0,0.041864,False


In [193]:
df_full_y

0        False
1        False
2        False
3        False
4        False
         ...  
55482    False
55483    False
55484    False
55485    False
55486    False
Name: is_anomaly, Length: 219644, dtype: bool

In [213]:
df_full_y.value_counts()

is_anomaly
False    215618
True       4026
Name: count, dtype: int64

In [27]:
y_true_test.value_counts()

is_anomaly
False    7002
True     4024
Name: count, dtype: int64

In [29]:
y_pred_test.value_counts()

is_anomaly_predicted
False    8921
True     2105
Name: count, dtype: int64

In [275]:
sklearn.metrics.accuracy_score(y_true, y_pred)

0.9492653654050166

In [30]:
sklearn.metrics.accuracy_score(y_true_test, y_pred_test)

0.5569562851442046

In [276]:
sklearn.metrics.precision_score(y_true, y_pred)

0.07437522420184145

In [31]:
sklearn.metrics.precision_score(y_true_test, y_pred_test)

0.2954869358669834

In [277]:
sklearn.metrics.recall_score(y_true, y_pred)

0.15457256461232605

In [32]:
sklearn.metrics.recall_score(y_true_test, y_pred_test)

0.15457256461232605

In [205]:
df_full

,time_stamp,sensor_0_avg,sensor_1_avg,sensor_2_avg,wind_speed_3_avg,wind_speed_4_avg,wind_speed_3_max,wind_speed_3_min,wind_speed_3_std,sensor_5_avg,...,sensor_48,sensor_49,sensor_50,sensor_51,sensor_52_avg,sensor_52_max,sensor_52_min,sensor_52_std,sensor_53_avg,is_anomaly
0,2022-07-28 13:20:00,31.0,152.0,48.7,3.9,3.9,8.0,0.6,0.9,70.5,...,0.0,0.0,-1185.0,-2090.0,0.4,2.6,0.0,0.8,34.0,False
1,2022-07-28 13:30:00,31.0,86.1,150.9,6.0,6.0,9.9,0.6,1.4,86.0,...,0.0,0.0,-1050.0,-1627.0,0.0,0.0,0.0,0.0,34.0,False
2,2022-07-28 13:40:00,31.0,115.2,69.6,6.3,6.3,10.6,0.8,1.3,86.0,...,0.0,0.0,-1043.0,-1624.0,0.0,0.0,0.0,0.0,34.0,False
3,2022-07-28 13:50:00,32.0,129.3,-29.1,6.0,5.9,12.4,1.7,1.4,13.6,...,-9540.0,0.0,40124.0,-9753.0,9.5,14.0,0.0,4.8,34.0,False
4,2022-07-28 14:00:00,32.0,137.7,26.4,7.1,6.9,13.7,1.7,1.7,-1.9,...,-25215.0,0.0,99360.0,-25215.0,13.1,14.9,10.8,1.3,35.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219639,2023-05-20 00:30:00,19.0,97.3,0.7,8.4,8.2,16.3,1.7,1.3,-2.2,...,-28066.0,0.0,159979.0,-28066.0,14.4,14.8,13.0,0.3,21.0,False
219640,2023-05-20 00:40:00,19.0,91.5,-5.0,9.3,8.9,16.0,3.7,1.3,-2.0,...,-28949.0,0.0,204068.0,-28949.0,14.7,14.9,14.1,0.1,21.0,False
219641,2023-05-20 00:50:00,19.0,83.3,-13.2,8.3,8.1,13.4,3.7,1.1,-2.2,...,-26837.0,0.0,155666.0,-26837.0,14.4,14.8,13.4,0.3,21.0,False
219642,2023-05-20 01:00:00,19.0,111.2,14.6,8.3,8.2,14.6,3.6,1.1,-2.2,...,-25319.0,0.0,160533.0,-25319.0,14.4,14.9,13.4,0.3,21.0,False


In [218]:
y_pred.value_counts()

is_anomaly_predicted
False    212531
True       7113
Name: count, dtype: int64